In [6]:
import xarray as xr
import numpy as np
import geospatialtools.gdal_tools as gdal_tools
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------
input_dir = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS/"
hb_grid_file = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/experiments/simulations/MSWX_3h_Snow_Project3_pp/postprocess/hrus.vrt"
output_dir = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS/HBGRID/"
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# List of NLDAS files
# --------------------------------------------------
files = [
    "precip.nc",
    "spfh.nc",
    "tair.nc",
    "lwdown.nc",
    "swdown.nc",
    "psurf.nc",
    "wind.nc"
]

# --------------------------------------------------
# Read HydroBlocks grid
# --------------------------------------------------
hb = gdal_tools.read_raster(hb_grid_file)
metad = gdal_tools.retrieve_metadata(hb_grid_file)

ny, nx = hb.shape
gt = metad["gt"]   # [xmin, dx, 0, ymax, 0, dy]

xmin = gt[0]
dx   = gt[1]
ymax = gt[3]
dy   = gt[5]

hb_lon = xmin + dx * (0.5 + np.arange(nx))
hb_lat = ymax + dy * (0.5 + np.arange(ny))

# xarray interpolation works best if latitude is increasing
if hb_lat[0] > hb_lat[-1]:
    hb_lat = hb_lat[::-1]

print("HB grid shape:", ny, nx)
print("HB lon range:", hb_lon.min(), hb_lon.max())
print("HB lat range:", hb_lat.min(), hb_lat.max())

# --------------------------------------------------
# Loop through each file
# --------------------------------------------------
for fname in files:
    fpath = os.path.join(input_dir, fname)

    if not os.path.exists(fpath):
        print(f"File not found, skipping: {fpath}")
        continue

    print("\n===================================")
    print("Processing:", fname)

    ds = xr.open_dataset(fpath, decode_times=False)

    varname = list(ds.data_vars)[0]
    da = ds[varname]

    print("Variable name:", varname)
    print("Original dims :", da.dims)
    print("Original shape:", da.shape)
    print("Coords        :", list(ds.coords))

    # detect coordinate names
    if "lon" in da.coords:
        lon_name = "lon"
    elif "longitude" in da.coords:
        lon_name = "longitude"
    else:
        raise ValueError(f"Could not find longitude coordinate in {fname}")

    if "lat" in da.coords:
        lat_name = "lat"
    elif "latitude" in da.coords:
        lat_name = "latitude"
    else:
        raise ValueError(f"Could not find latitude coordinate in {fname}")

    # make sure source latitude is increasing
    if da[lat_name][0] > da[lat_name][-1]:
        da = da.sortby(lat_name)

    # interpolate to HB grid
    da_hb = da.interp(
        {lon_name: hb_lon, lat_name: hb_lat},
        method="linear"
    )

    print("Interpolated shape:", da_hb.shape)

    # save
    da_hb.name = varname
    ds_out = da_hb.to_dataset()

    outname = fname.replace(".nc", "_HBgrid.nc")
    outpath = os.path.join(output_dir, outname)

    ds_out.to_netcdf(outpath)

    print("Saved to:", outpath)
    print("Before min/max:", float(np.nanmin(da.values)), float(np.nanmax(da.values)))
    print("After  min/max:", float(np.nanmin(da_hb.values)), float(np.nanmax(da_hb.values)))

    ds.close()

print("\nDone.")

HB grid shape: 1801 1801
HB lon range: -106.44999999999882 -104.94999999999888
HB lat range: 35.04999999999927 36.549999999999216

Processing: precip.nc
Variable name: precip
Original dims : ('t', 'lat', 'lon')
Original shape: (32128, 16, 16)
Coords        : ['lon', 'lat', 't']


: 

: 

In [9]:
import xarray as xr
import numpy as np
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------
input_dir = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS/"
template_file = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/precip.nc"
output_dir = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS_on_HBmetgrid/"
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# NLDAS files to regrid
# --------------------------------------------------
files = [
    "precip.nc",
    "spfh.nc",
    "tair.nc",
    "lwdown.nc",
    "swdown.nc",
    "psurf.nc",
    "wind.nc"
]

# --------------------------------------------------
# Read HydroBlocks meteorology grid from template
# --------------------------------------------------
ds_tmp = xr.open_dataset(template_file, decode_times=False)

target_lat = ds_tmp["lat"].values
target_lon = ds_tmp["lon"].values

print("Target grid shape:", len(target_lat), len(target_lon))
print("Target lat range:", target_lat.min(), target_lat.max())
print("Target lon range:", target_lon.min(), target_lon.max())

ds_tmp.close()

# --------------------------------------------------
# Loop through each NLDAS file
# --------------------------------------------------
for fname in files:
    fpath = os.path.join(input_dir, fname)

    if not os.path.exists(fpath):
        print(f"Missing, skipping: {fpath}")
        continue

    print("\n==============================")
    print("Processing:", fname)

    # open without decoding time because units are '3hr since ...'
    ds = xr.open_dataset(fpath, decode_times=False)

    varname = list(ds.data_vars)[0]
    da = ds[varname]

    print("Variable:", varname)
    print("Original shape:", da.shape)
    print("Original dims:", da.dims)

    if "lat" not in da.coords or "lon" not in da.coords:
        raise ValueError(f"{fname} must have lat/lon coordinates")

    # make source latitude increasing
    if da["lat"][0] > da["lat"][-1]:
        da = da.sortby("lat")

    # interpolate to HydroBlocks forcing grid
    da_out = da.interp(
        lat=target_lat,
        lon=target_lon,
        method="linear"
    )

    # match HydroBlocks file structure
    da_out = da_out.astype("float32")
    da_out = da_out.transpose("t", "lat", "lon")
    da_out.attrs = da.attrs.copy()

    ds_out = da_out.to_dataset(name=varname)

    # preserve coordinate attrs if present
    for coord in ["t", "lat", "lon"]:
        if coord in ds.coords and coord in ds_out.coords:
            ds_out[coord].attrs = ds[coord].attrs.copy()

    outpath = os.path.join(output_dir, fname)

    # force float32 in the saved netcdf
    encoding = {varname: {"dtype": "float32"}}
    ds_out.to_netcdf(outpath, mode="w", encoding=encoding)

    print("Interpolated shape:", da_out.shape)
    print("Saved:", outpath)
    print("Output dtype in memory:", ds_out[varname].dtype)

    ds.close()

print("\nDone.")

Target grid shape: 16 16
Target lat range: 35.099998474121094 36.5999755859375
Target lon range: -106.39999771118164 -104.90002059936523

Processing: precip.nc
Variable: precip
Original shape: (32128, 16, 16)
Original dims: ('t', 'lat', 'lon')
Interpolated shape: (32128, 16, 16)
Saved: /scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS_on_HBmetgrid/precip.nc
Output dtype in memory: float32

Processing: spfh.nc
Variable: spfh
Original shape: (32128, 16, 16)
Original dims: ('t', 'lat', 'lon')
Interpolated shape: (32128, 16, 16)
Saved: /scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS_on_HBmetgrid/spfh.nc
Output dtype in memory: float32

Processing: tair.nc
Variable: tair
Original shape: (32128, 16, 16)
Original dims: ('t', 'lat', 'lon')
Interpolated shape: (32128, 16, 16)
Saved: /scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/ci

In [13]:
ds_check = xr.open_dataset(
    "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS_on_HBmetgrid_float32/precip_float32.nc",
    decode_times=False
)
print(ds_check)
print(ds_check["precip"].dtype)

<xarray.Dataset>
Dimensions:  (t: 32128, lat: 16, lon: 16)
Coordinates:
  * t        (t) float64 0.0 1.0 2.0 3.0 ... 3.212e+04 3.213e+04 3.213e+04
  * lat      (lat) float64 35.1 35.2 35.3 35.4 35.5 ... 36.2 36.3 36.4 36.5 36.6
  * lon      (lon) float64 -106.4 -106.3 -106.2 -106.1 ... -105.1 -105.0 -104.9
Data variables:
    precip   (t, lat, lon) float32 ...
float32


In [11]:
ds_check = xr.open_dataset(
    "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS_on_HBmetgrid/precip.nc",
    decode_times=False
)
print(ds_check)
print(ds_check["precip"].dtype)

<xarray.Dataset>
Dimensions:  (t: 32128, lat: 16, lon: 16)
Coordinates:
  * t        (t) float64 0.0 1.0 2.0 3.0 ... 3.212e+04 3.213e+04 3.213e+04
  * lat      (lat) float64 35.1 35.2 35.3 35.4 35.5 ... 36.2 36.3 36.4 36.5 36.6
  * lon      (lon) float64 -106.4 -106.3 -106.2 -106.1 ... -105.1 -105.0 -104.9
Data variables:
    precip   (t, lat, lon) float64 ...
float64


In [10]:
import xarray as xr

ds_old = xr.open_dataset("/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/precip.nc", decode_times=False)
ds_new = xr.open_dataset("/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/backup_forcing/NLDAS_on_HBmetgrid/precip.nc", decode_times=False)

print("OLD:")
print(ds_old)
print("\nNEW:")
print(ds_new)

OLD:
<xarray.Dataset>
Dimensions:  (lon: 16, lat: 16, t: 32128)
Coordinates:
  * lon      (lon) float64 -106.4 -106.3 -106.2 -106.1 ... -105.1 -105.0 -104.9
  * lat      (lat) float64 35.1 35.2 35.3 35.4 35.5 ... 36.2 36.3 36.4 36.5 36.6
  * t        (t) float64 0.0 1.0 2.0 3.0 ... 3.212e+04 3.213e+04 3.213e+04
Data variables:
    precip   (t, lat, lon) float32 ...

NEW:
<xarray.Dataset>
Dimensions:  (t: 32128, lat: 16, lon: 16)
Coordinates:
  * t        (t) float64 0.0 1.0 2.0 3.0 ... 3.212e+04 3.213e+04 3.213e+04
  * lat      (lat) float64 35.1 35.2 35.3 35.4 35.5 ... 36.2 36.3 36.4 36.5 36.6
  * lon      (lon) float64 -106.4 -106.3 -106.2 -106.1 ... -105.1 -105.0 -104.9
Data variables:
    precip   (t, lat, lon) float64 ...


In [2]:
import os

print(os.path.exists("/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/1/hrus.vrt"))
print(os.path.exists("/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/hrus.vrt"))
print(os.path.exists("/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/data/snow_experiment/data/cids/1/cids.vrt"))

False
False
False


In [1]:
from osgeo import gdal
import os
import glob

# Paths
#modis_dir = "/projects/battobrah@xsede.org/Remote_Sensing_Validation_data/MODIS_tsurf"
modis_dir = "/projects/battobrah@xsede.org/Remote_Sensing_Validation_data/MODIS_LST_NEW"
out_dir = "/scratch/alpine/battobrah@xsede.org/MODIS_on_HBgrid_new"
hb_template = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/experiments/simulations/MSWX_3h_Snow_Project3_pp_downscale_new/postprocess/hrus.vrt"

os.makedirs(out_dir, exist_ok=True)

# Get target grid from HRU template
tmpl = gdal.Open(hb_template)
gt = tmpl.GetGeoTransform()
proj = tmpl.GetProjection()
nx = tmpl.RasterXSize
ny = tmpl.RasterYSize

# Compute extent
xmin = gt[0]
xmax = gt[0] + nx * gt[1]
ymax = gt[3]
ymin = gt[3] + ny * gt[5]

print("Target grid:")
print("Extent:", xmin, ymin, xmax, ymax)
print("Resolution:", gt[1], gt[5])
print("Size:", nx, ny)

# Loop through MODIS files
files = sorted(glob.glob(os.path.join(modis_dir, "*.tif")))

for f in files:
    fname = os.path.basename(f)
    out_file = os.path.join(out_dir, fname)

    print(f"Processing: {fname}")

    gdal.Warp(
        out_file,
        f,
        format="GTiff",
        dstSRS=proj,
        outputBounds=(xmin, ymin, xmax, ymax),
        xRes=gt[1],
        yRes=abs(gt[5]),
        resampleAlg="bilinear",
        dstNodata=-9999
    )

print("Done.")

Target grid:
Extent: -106.45041666666549 35.049583333332606 -104.94958333333221 36.55041666666588
Resolution: 0.0008333333333333 -0.0008333333333333
Size: 1801 1801
Processing: MOD11A1_monthly_mean_2016_01.tif
Processing: MOD11A1_monthly_mean_2016_02.tif
Processing: MOD11A1_monthly_mean_2016_03.tif
Processing: MOD11A1_monthly_mean_2016_04.tif
Processing: MOD11A1_monthly_mean_2016_05.tif
Processing: MOD11A1_monthly_mean_2016_06.tif
Processing: MOD11A1_monthly_mean_2016_07.tif
Processing: MOD11A1_monthly_mean_2016_08.tif
Processing: MOD11A1_monthly_mean_2016_09.tif
Processing: MOD11A1_monthly_mean_2016_10.tif
Processing: MOD11A1_monthly_mean_2016_11.tif
Processing: MOD11A1_monthly_mean_2016_12.tif
Processing: MOD11A1_monthly_mean_2017_01.tif
Processing: MOD11A1_monthly_mean_2017_02.tif
Processing: MOD11A1_monthly_mean_2017_03.tif
Processing: MOD11A1_monthly_mean_2017_04.tif
Processing: MOD11A1_monthly_mean_2017_05.tif
Processing: MOD11A1_monthly_mean_2017_06.tif
Processing: MOD11A1_month

In [2]:
from osgeo import gdal
import numpy as np

test_file = "/scratch/alpine/battobrah@xsede.org/MODIS_on_HBgrid_new/MOD11A1_monthly_mean_2016_01.tif"

ds = gdal.Open(test_file)
band = ds.GetRasterBand(1)

data = band.ReadAsArray()

print("Shape:", data.shape)
print("Min:", np.nanmin(data))
print("Max:", np.nanmax(data))
print("NoData:", band.GetNoDataValue())
print("Scale:", band.GetScale())
print("Offset:", band.GetOffset())

Shape: (1801, 1801)
Min: -9.994806728268811
Max: 15.253312595915567
NoData: -9999.0
Scale: None
Offset: None


In [3]:
import netCDF4 as nc
import numpy as np

hb_file = "/scratch/alpine/battobrah@xsede.org/HydroBlocks_Enrico_dev/experiments/simulations/MSWX_3h_Snow_Project3_pp_downscale_new/output_data/1/2016-01-01.nc"

with nc.Dataset(hb_file) as ds:
    trad = ds.groups["data"].variables["trad"][:]
    print("Shape:", trad.shape)
    print("Min:", np.nanmin(trad))
    print("Max:", np.nanmax(trad))

Shape: (8744, 1000)
Min: 238.01562
Max: 332.01562
